In [22]:
# use dataset 1 and Balanced Random Forest which be choosed by iM-seeker
import pandas as pd
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score, average_precision_score, precision_score, recall_score, f1_score
import torch
import torch.nn as nn
import pickle

# 加载数据
def load_data(file_positive, file_negative):
    # 读取正样本和负样本
    positive_samples = pd.read_csv(file_positive, sep='\t', header=None)
    negative_samples = pd.read_csv(file_negative, sep='\t', header=None)
    
    # 合并数据集
    data = pd.concat([positive_samples, negative_samples], axis=0)
    
    # 为正样本和负样本分配标签
    data['label'] = [1.] * len(positive_samples) + [0.] * len(negative_samples)
    
    # 分离特征和标签
    X = data.iloc[:, :-1]
    y = data['label']
    
    return X, y

# 训练集和测试集文件路径
train_positive_file = './33_feats/positive_train_samples_dataset1.csv'
train_negative_file = './33_feats/negative_train_samples_dataset1.csv'
test_positive_file = './33_feats/positive_test_samples_dataset1.csv'
test_negative_file = './33_feats/negative_test_samples_dataset1.csv'

In [23]:
# 加载训练集和测试集
X_train, y_train = load_data(train_positive_file, train_negative_file)
X_test, y_test = load_data(test_positive_file, test_negative_file)

# 初始化 BalancedRandomForestClassifier
'''
{'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': None, 
'criterion': 'gini', 'max_depth': 32, 'max_features': 'sqrt', 
'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 
'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 
'n_estimators': 200, 'n_jobs': None, 'oob_score': False, 'random_state': 0, 
'replacement': False, 'sampling_strategy': 'all', 'verbose': 0, 'warm_start': False}
'''
brf = BalancedRandomForestClassifier(n_estimators=200, max_depth=32, max_features='sqrt', random_state=0)

# 训练模型
brf.fit(X_train, y_train)

d:\miniconda3\envs\imotif\Lib\site-packages\imblearn\ensemble\_forest.py:577: FutureWarning: The default of `sampling_strategy` will change from `'auto'` to `'all'` in version 0.13. This change will follow the implementation proposed in the original paper. Set to `'all'` to silence this warning and adopt the future behaviour.
  warn(
d:\miniconda3\envs\imotif\Lib\site-packages\imblearn\ensemble\_forest.py:589: FutureWarning: The default of `replacement` will change from `False` to `True` in version 0.13. This change will follow the implementation proposed in the original paper. Set to `True` to silence this warning and adopt the future behaviour.
  warn(
d:\miniconda3\envs\imotif\Lib\site-packages\imblearn\ensemble\_forest.py:601: FutureWarning: The default of `bootstrap` will change from `True` to `False` in version 0.13. This change will follow the implementation proposed in the original paper. Set to `False` to silence this warning and adopt the future behaviour.
  warn(


BalancedRandomForestClassifier(max_depth=32, n_estimators=200, random_state=0)

In [24]:
# 使用模型进行预测
y_pred = brf.predict(X_test)

# 计算准确率
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy}')

# 计算交叉熵损失
cross_entropy_loss = log_loss(y_test, brf.predict_proba(X_test)[:, 1])
print(f'Cross-Entropy Loss: {cross_entropy_loss}')

# 将预测结果转换为PyTorch的Tensor格式
# 对于二分类问题，我们需要将预测的概率转换为logits
# 我们使用logit = log(p/(1-p))，其中p是预测为正类的概率
y_pred_probs = brf.predict_proba(X_test)
y_pred_probs = torch.tensor(y_pred_probs, dtype=torch.float32)
y_pred_probs -= 0.5

# 真实标签也需要转换为Tensor格式
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.long)

# 初始化交叉熵损失函数
criterion = nn.CrossEntropyLoss()

# 由于nn.CrossEntropyLoss包括了softmax操作，我们不需要手动计算softmax
# 直接计算交叉熵损失
cross_entropy_loss = criterion(y_pred_probs[:2], y_test_tensor[:2])
print(f'Cross-Entropy Loss: {cross_entropy_loss.item()}')

# 计算AUROC
auroc = roc_auc_score(y_test, brf.predict_proba(X_test)[:, 1])
print(f'AUROC: {auroc}')

# 计算AUPRC
auprc = average_precision_score(y_test, brf.predict_proba(X_test)[:, 1])
print(f'AUPRC: {auprc}')

# 计算Precision, Recall, F1-score
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
print(f'Precision: {precision}')
print(f'Recall: {recall}')
print(f'F1-score: {f1}')

Accuracy: 0.7865497076023392
Cross-Entropy Loss: 0.4856150731453027
Cross-Entropy Loss: 0.6006501913070679
AUROC: 0.8819901382254176
AUPRC: 0.879701536392667
Precision: 0.8834714773281326
Recall: 0.6610726012404232
F1-score: 0.7562604340567612


n_estimators=100
>Accuracy: 0.783625730994152
Cross-Entropy Loss: 0.484681240386728
Cross-Entropy Loss: 0.6338090896606445
AUROC: 0.8793434816999036
AUPRC: 0.8761863781451811
Precision: 0.8836865450961064
Recall: 0.6541408245165998
F1-score: 0.7517819706498952

In [25]:
# features importance
features33_names = [
    "[1] C-tract length",
    "[2] iM length",
    "[3] loop length",
    "[4] middle loop length",
    "[5] longest side loop length",
    "[6] shortest side loop length",
    "[7] sum of two side loops",
    "[8] longest loop length",
    "[9] shortest loop length",
    "[10] A density in iMs",
    "[11] C density in iMs",
    "[12] G density in iMs",
    "[13] T density in iMs",
    "[14] A density in loops",
    "[15] C density in loops",
    "[16] G density in loops",
    "[17] T density in loops",
    "[18] A density in middle loop", 
    "[19] C density in middle loop",
    "[20] G density in middle loop",
    "[21] T density in middle loop",
    "[22] A density in longest side loop",
    "[23] C density in longest side loop",
    "[24] G density in longest side loop",
    "[25] T density in longest side loop",
    "[26] A density in shortest side loop",
    "[27] C density in shortest side loop",
    "[28] G density in shortest side loop",
    "[29] T density in shortest side loop",
    "[30] A density in two side loops",
    "[31] C density in two side loops",
    "[32] G density in two side loops",
    "[33] T density in two side loops"
]

# importances = brf.feature_importances_
# indices = importances.argsort()[::-1]
# print('Feature ranking:')
# for f in range(X_train.shape[1]):
#     print(f'{f+1}. {features33_names[indices[f]]}: {importances[indices[f]]}')

# compute Pearson correlation coefficient between 33 features and prediction probabilities
from scipy.stats import pearsonr
correlations = list()
for i in range(X_train.shape[1]):
    corr, _ = pearsonr(X_test.iloc[:, i], y_pred_probs[:, 1])
    correlations.append([features33_names[i], corr])
correlations.sort(key=lambda x: x[1], reverse=True)
for i in range(len(correlations)):
    print(f"{i+1}th: {correlations[i]}")

1th: ['[16] G density in loops', 0.4864948537605616]
2th: ['[12] G density in iMs', 0.45755248043803076]
3th: ['[32] G density in two side loops', 0.4245931038773447]
4th: ['[28] G density in shortest side loop', 0.4133930815806446]
5th: ['[31] C density in two side loops', 0.37193312559402986]
6th: ['[20] G density in middle loop', 0.3697021662558323]
7th: ['[15] C density in loops', 0.3532494910125147]
8th: ['[11] C density in iMs', 0.3463354417500223]
9th: ['[23] C density in longest side loop', 0.32270460983590477]
10th: ['[24] G density in longest side loop', 0.31647211138648157]
11th: ['[27] C density in shortest side loop', 0.19993186181516162]
12th: ['[19] C density in middle loop', 0.15992659518664007]
13th: ['[1] C-tract length', 0.14846556926775634]
14th: ['[4] middle loop length', 0.06734266862723179]
15th: ['[2] iM length', 0.048232381628701]
16th: ['[3] loop length', 0.02096496443324971]
17th: ['[9] shortest loop length', 0.012375738144403266]
18th: ['[8] longest loop len

In [27]:
## output to file
# with open("correlations.csv", "w") as f:
#     for i in range(33):
#         f.write(f"{i+1},{correlations[i][0]},{correlations[i][1]}\n")